# 模組 7 · 模型可解釋性
## SHAP 基礎、SHAP × sklearn、特徵（波長）選擇

模型不只要準，還要能解釋。學會用 SHAP 找出重要波長，並用 CARS/MC-UVE 做波長選擇。

**對應官方範例**：`examples/user/07_explainability/`（U01–U03）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · SHAP 基礎：解釋模型預測  ★★☆☆☆

**SHAP** 量化每個波長對預測的貢獻。對 NIRS 特別有用：光譜圖找重要區段、瀑布圖解釋單筆、蜂群圖看模式。高維光譜可用分箱讓結果更易讀。需 `pip install shap`。

In [ ]:
# SHAP 為可選相依，第一次需安裝
import subprocess, sys
try:
    import shap
except ImportError:
    print("📦 安裝 shap …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import ShuffleSplit
from nirs4all.operators.transforms import Gaussian, SavitzkyGolay, StandardNormalVariate

result = nirs4all.run(
    pipeline=[{"y_processing": MinMaxScaler()}, MinMaxScaler(),
              Gaussian, SavitzkyGolay, StandardNormalVariate,
              ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
              {"model": PLSRegression(n_components=16)}],
    dataset="sample_data/regression", name="ShapBasics",
    verbose=1, save_artifacts=True)
print("\n模型訓練完成，可用 nirs4all.explain() 進行 SHAP 分析。")
print("💡 explain 會輸出光譜圖、瀑布圖、蜂群圖到 workspace。")

> ✍️ **練習**：找出貢獻最大的波長區段，對照已知化學吸收帶（如水分約 1450 / 1940 nm）。

---
## U02 · SHAP × sklearn 包裝器  ★★★☆☆

用 `NIRSPipeline` 包裝後可直接接 SHAP 各式 explainer：`KernelExplainer`（任何模型，最慢）、`TreeExplainer`（樹模型，快 10–100 倍）、`LinearExplainer`（線性）。用 `shap.kmeans()` 做高效背景集。

In [ ]:
import subprocess, sys
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"]); import shap
import numpy as np, nirs4all
from nirs4all.sklearn import NIRSPipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import ShuffleSplit
from nirs4all.operators.transforms import StandardNormalVariate

result = nirs4all.run(
    pipeline=[StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
              ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
              {"model": PLSRegression(n_components=10)}],
    dataset="sample_data/regression", name="ShapSklearn", verbose=1)

pipe = NIRSPipeline.from_result(result)
print("\n已建立 NIRSPipeline，可接 SHAP：")
print("  background = shap.kmeans(X, 10)")
print("  explainer = shap.KernelExplainer(pipe.predict, background)")
print("  shap_values = explainer.shap_values(X_test)")
print("  importance = np.mean(np.abs(shap_values), axis=0)  # 各波長重要性")

> ✍️ **練習**：對樹模型改用 `TreeExplainer`（記得先 `pipe.transform(X)`），比較它與 KernelExplainer 的速度。

---
## U03 · 特徵選擇：CARS 與 MC-UVE  ★★★☆☆

並非所有波長都有用。**CARS**（競爭式自適應重取樣）與 **MC-UVE**（蒙地卡羅無資訊變數消除）挑出最具資訊的波長，提升精簡度與可解釋性。CARS 可直接放進 pipeline。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import FirstDerivative, StandardNormalVariate

# 比較：全波長 vs CARS 選擇後
results = {}
# 全波長
r_full = nirs4all.run(
    pipeline=[{"y_processing": MinMaxScaler()},
              {"feature_augmentation": [FirstDerivative, StandardNormalVariate]},
              MinMaxScaler(),
              ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
              PLSRegression(n_components=10)],
    dataset="sample_data/regression", name="FullWavelength", verbose=0)
print("全波長 RMSE:", round(r_full.best_rmse, 4))

# CARS 波長選擇
try:
    from nirs4all.operators.selection import CARS
    r_cars = nirs4all.run(
        pipeline=[{"y_processing": MinMaxScaler()},
                  {"feature_augmentation": [FirstDerivative, StandardNormalVariate]},
                  CARS(n_components=10, n_sampling_runs=10,
                       n_variables_ratio_end=0.2, cv_folds=5, random_state=42),
                  MinMaxScaler(),
                  ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
                  PLSRegression(n_components=10)],
        dataset="sample_data/regression", name="CARS", verbose=0)
    print("CARS 選擇後 RMSE:", round(r_cars.best_rmse, 4))
except Exception as e:
    print("（CARS 不可用，略過）", e)

> ✍️ **練習**：CARS 用更少波長能否維持精度？比較兩者 RMSE 與所用波長數，評估精簡效益。